In [1]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [2]:
## spatial join
# target_features = ?
# join_features = ?
# output_features = os.path.join(gdb, ?)

# fieldmappings = arcpy.FieldMappings()
# fieldmappings.addTable(target_features)
# fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
# sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [3]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

## Add new Spaces to parcels

In [4]:
p1 = pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels')
p2 = pd.DataFrame.spatial.from_featureclass(r'E:\Projects\REMM-Job-Space-Calculation\2-Allocate-Spaces\Outputs\_05_Allocate_TAZ_Spaces.gdb\parcels_with_job_spaces')


p1['parcel_id'] = p1['parcel_id'].astype('Int32')
p2['parcel_id'] = p2['parcel_id'].astype('Int32')

In [5]:

wf_map = p2.dropna(subset=['parcel_id']).set_index('parcel_id')['job_spaces'].to_dict()

p1['job_spaces'] = 0
p1['job_spaces'] = p1['parcel_id'].map(wf_map).combine_first(p1['job_spaces'])


In [6]:
# output for h5
p1.drop(['SHAPE', 'OBJECTID'], axis=1).to_csv(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\parcels_20260131.csv",  index=False) 

In [7]:
# replace current parcels with this if all looks okay
p1.spatial.to_featureclass(location=r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels_NEW',sanitize_columns=False) 
del p1, p2, wf_map